# **Семинар 2.** Векторные представления слов. Word2Vec

В прошлый раз мы разобрались с тем, как представить текст в виде вектора чисел фиксированной размерности с помощью Bag-of-Words. Но был у нашего метода и существенный изъян — полученные векторы ничего не знали о смысле отдельных слов: Солнце было так же далеко от Луны, как Луна от яблока.

Сегодня попробуем это исправить и придать словам (а если получится, то и всему происходящему) капельку смысла.

## **1.1. Дистрибутивная гипотеза**

Но для начала необходимо понять: а что же такое вообще смысл? Давайте для этого сыграем в игру. Изучите тексты ниже и попробуйте угадать, что означает слово **мауби**:

- **Мауби** любят как взрослые, так и дети.
- Глоток **мауби** отлично прохлаждает в жару.

Получилось? Отлично! Мы только что наглядно продемонстрировали то, что называется **дистрибутивной гипотезой** (англ. *distributional hypothesis*):

<div style="border-left: 4px solid dodgerblue; padding: 12px 12px 12px 24px; margin: 12px 0; color: dodgerblue; display: flex; align-items: center;">
    <strong>Слова, часто встречающиеся в похожих контекстах, имеют похожий смысл.</strong>
</div>

Иными словами, смысл слова во многом определяется его контекстом.

## **1.2. Word2Vec**

Вооружившись этой невероятно простой, но очень полезной идеей, к которой мы еще не раз вернемся в рамках курса, попробуем построить векторные представления слов с помощью **Word2Vec**.

[Подробное и очень красочное изложение теории Word2Vec](https://lena-voita.github.io/nlp_course/word_embeddings.html#w2v_idea) можно найти в замечательном курсе [NLP Course For You](https://lena-voita.github.io/nlp_course.html).

In [ ]:
from typing import Literal

import pandas as pd

FIELDS = [
    "Биология",
    "Геология",
    "Математика",
    "Медицина и здравоохранение",
    "Психология",
    "Социология",
    "Физика",
    "Философия",
    "Химия",
    "Языкознание",
]


def load_ru_sci_bench(split: Literal["train", "test"]) -> pd.DataFrame:
    data = pd.read_json(
        f"https://huggingface.co/datasets/ai-forever/ru-scibench-grnti-classification/resolve/main/{split}.jsonl",
        lines=True,
    )

    return data[data["label_text"].isin(FIELDS)]


train = load_ru_sci_bench("train")
train.head()

In [ ]:
len(train)

In [ ]:
train["label_text"].value_counts()

In [ ]:
from nltk.tokenize import WordPunctTokenizer
from tqdm import tqdm

tokenizer = WordPunctTokenizer()


def tokenize(text: str) -> list[str]:
    return tokenizer.tokenize(text.lower())


tokenized_train = []

for text in tqdm(train["text"]):
    tokenized_train.append(tokenize(text))

" ".join(tokenized_train[0][:10])

In [ ]:
from gensim.models import Word2Vec

wv = Word2Vec(
    sentences=tokenized_train,
    vector_size=128,
    window=5,
    min_count=5,
    max_vocab_size=100_000,
    seed=42,
    sg=0,
    negative=25,
    epochs=10,
).wv

type(wv)

In [ ]:
wv.vector_size

In [ ]:
wv.vectors.shape

In [ ]:
wv.key_to_index["химия"]

In [ ]:
wv.index_to_key[:10]

In [ ]:
wv["химия"]

In [ ]:
wv.similarity("химия", "биология")

In [ ]:
wv.most_similar("метод", topn=5)

## **1.3. Классификация текстов**

Мы обучили Word2Vec и готовы использовать его для классификации текстов. Осталось только разобраться, как из векторов отдельных токенов получить вектор всего текста. Самый простой способ — просто усреднить их (англ. *mean pooling*).

In [ ]:
import numpy as np

tokenized_text = tokenize("Математика — царица всех наук")

v1 = np.mean([wv[token] for token in tokenized_text if token in wv], axis=0)
v1

Тот же результат можно получить с помощью Gensim:

In [ ]:
v2 = wv.get_mean_vector(
    tokenized_text,
    pre_normalize=False,
    post_normalize=False,
)
v2

In [ ]:
np.allclose(v1, v2)

Обратите внимание, что Gensim автоматически проигнорировал OOV-токены:

In [ ]:
"царица" in wv

In [ ]:
from numpy.typing import NDArray


def encode(tokenized: list[list[str]]) -> NDArray[np.float32]:
    return np.stack(
        [
            wv.get_mean_vector(
                tokens,
                pre_normalize=False,
                post_normalize=False,
            )
            for tokens in tokenized
        ]
    )


X_train = encode(tokenized_train)
X_train.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
pipeline.fit(X_train, train["label_text"])

In [ ]:
test = load_ru_sci_bench("test")
test.head()

In [ ]:
tokenized_test = [tokenize(text) for text in test["text"]]

X_test = encode(tokenized_test)
X_test.shape

In [ ]:
from sklearn.metrics import classification_report

preds = pipeline.predict(X_test)
print(classification_report(test["label_text"], preds, digits=3, zero_division=0))

При обычном усреднении все векторы слов вносят одинаковый вклад в итоговое представление текста. Давайте вместо этого взвесим их с помощью TF-IDF: тогда векторы более важных для данного текста слов будут вносить больший вклад в итоговый вектор.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=False,
    preprocessor=None,
    tokenizer=lambda tokens: tokens,
    token_pattern=None,
    vocabulary=wv.key_to_index,
)

tfidf_train = tfidf.fit_transform(tokenized_train)
tfidf_train

In [ ]:
X_train_weighted = tfidf_train @ wv.vectors
X_train_weighted.shape

In [ ]:
pipeline.fit(X_train_weighted, train["label_text"])

In [ ]:
tfidf_test = tfidf.transform(tokenized_test)

X_test_weighted = tfidf_test @ wv.vectors
X_test_weighted.shape

In [ ]:
from sklearn.metrics import classification_report

preds = pipeline.predict(X_test_weighted)
print(classification_report(test["label_text"], preds, digits=3, zero_division=0))

In [ ]:
import plotly.express as px
from umap import UMAP

umap1, umap2 = UMAP(random_state=42, n_jobs=1).fit_transform(X_test_weighted).T

fig = px.scatter(
    pd.DataFrame({"UMAP1": umap1, "UMAP2": umap2, "Field": test["label_text"]}),
    x="UMAP1",
    y="UMAP2",
    color="Field",
)
fig.show()

## **1.3. Свойства Word2Vec**

Одно свойство Word2Vec напрямую следует из дистрибутивной гипотезы и самой постановки задачи обучения — похожие по смыслу слова получают похожие векторы. Его мы уже проверили.

Но у получившегося пространства есть и другие, куда более удивительные свойства. Мы не закладывали их в модель явно, однако они возникли сами в процессе обучения.

Давайте возьмем для экспериментов предобученные на текстах Common Crawl и Wikipedia векторы [fastText](https://fasttext.cc/docs/en/crawl-vectors.html).

**Примечание:** строго говоря, fastText — это не классический Word2Vec, а его модификация, в которой представление слова строится с учетом символьных n-грамм. Более детальное описание метода можно найти в статье [*Enriching Word Vectors with Subword Information*](https://arxiv.org/abs/1607.04606).

In [ ]:
# %%bash
# cd data

# wget -q https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.ru.300.bin.gz
# wget -q https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.en.300.bin.gz

# gunzip *.gz

При желании можно воспользоваться официальной библиотекой `fasttext` [[GitHub](https://github.com/facebookresearch/fastText)].

In [ ]:
import fasttext

wv = fasttext.load_model("data/cc.ru.300.bin")

In [ ]:
wv.get_word_vector("луна")

In [ ]:
wv.get_nearest_neighbors("луна", k=5)

Но мы продолжим работать с `gensim`.

In [ ]:
from gensim.models.fasttext import load_facebook_vectors

wv = load_facebook_vectors("data/cc.ru.300.bin")

In [ ]:
wv.similar_by_key("принц", topn=5)

In [ ]:
v = wv.get_vector("королева", norm=True) - wv.get_vector("король", norm=True) + wv.get_vector("принц", norm=True)
wv.similar_by_vector(v, topn=5)

In [ ]:
v = wv.get_vector("женщина", norm=True) - wv.get_vector("мужчина", norm=True) + wv.get_vector("кот", norm=True)
wv.similar_by_vector(v, topn=5)

Для чуть более удобного поиска можно использовать метод `most_similar` с параметрами `positive` и `negative`: он дополнительно фильтрует исходные слова из результатов.

In [ ]:
wv.most_similar(
    positive=["женщина", "кот"],
    negative=["мужчина"],
    topn=5,
)

Получается, что в хорошо обученном пространстве векторов существуют почти линейные смысловые и грамматические отношения между словами.

Теперь подумаем вот о чем. В разных языках одни и те же по смыслу слова, скорее всего, используются в похожих контекстах. Вряд ли, если в русском языке Солнце всходит над горизонтом, то в английском оно входит в состав шарлотки.

Значит, пространства Word2Vec, независимо обученные для двух языков, должны обладать похожей внутренней структурой: взаимное расположение слов и отношения между ними должны во многом сохраняться от языка к языку. И чем ближе и родственнее два языка, тем большего сходства между их пространствами мы можем ожидать.

Проблема лишь в том, что одно пространство может быть смещено, повернуто, растянуто или сжато относительно другого. Попробуем научиться отображать одно пространство в другое.

Пусть нам известны $n$ пар соответствий слов и их переводов (например, солнце — sun, яблоко — apple). Обозначим векторные представления слов в $i$-й паре как $x_i \in \mathbb{R}^d$ и $y_i \in \mathbb{R}^d$. Тогда в простейшем случае мы можем найти матрицу $W \in \mathbb{R}^{d \times d}$, которая преобразует векторы первого языка так, чтобы они оказались как можно ближе к соответствующим векторам второго:

$$
\min_W \frac{1}{n}\sum_{i=1}^{n} \ell(Wx_i, y_i),
$$

где $\ell$ — функция потерь. Например, можно использовать обычную квадратичную ошибку:

$$
\ell(x, y) = \|x-y\|_2^2.
$$

Иными словами, мы просто обучим линейное отображение из одного пространства эмбеддингов в другое.

Для нашего эксперимента воспользуемся данными проекта MUSE [[GitHub](https://github.com/facebookresearch/MUSE), [arXiv](https://arxiv.org/abs/1710.04087)].

In [ ]:
# %%bash
# wget https://dl.fbaipublicfiles.com/arrival/dictionaries/ru-en.0-5000.txt -P data
# wget https://dl.fbaipublicfiles.com/arrival/dictionaries/ru-en.5000-6500.txt -P data

In [ ]:
!head data/ru-en.0-5000.txt

In [ ]:
with open("data/ru-en.0-5000.txt") as file:
    muse_train = [line.split() for line in file]

muse_train[:5]

In [ ]:
with open("data/ru-en.5000-6500.txt") as file:
    muse_test = [line.split() for line in file]

muse_test[:5]

In [ ]:
ru_wv = wv  # alias
en_wv = load_facebook_vectors("data/cc.en.300.bin")

In [ ]:
X_train, Y_train = [], []

for ru_word, en_word in tqdm(muse_train):
    X_train.append(ru_wv.get_vector(ru_word, norm=True))
    Y_train.append(en_wv.get_vector(en_word, norm=True))

X_train, Y_train = np.stack(X_train), np.stack(Y_train)
X_train.shape, Y_train.shape

In [ ]:
from sklearn.linear_model import LinearRegression

linreg = LinearRegression(fit_intercept=False)
linreg.fit(X_train, Y_train)

W = linreg.coef_
W.shape

In [ ]:
ru_v = ru_wv.get_vector("вчера", norm=True)
en_v = W @ v
en_wv.similar_by_vector(en_v)

In [ ]:
from random import Random

from sklearn.decomposition import PCA


rng = Random(42)

ru_words, ru_vectors = [], []
en_words, en_vectors = [], []

N = 10

for ru_word, en_word in rng.sample(muse_test, k=N):
    ru_words.append(ru_word), ru_vectors.append(ru_wv.get_vector(ru_word, norm=True))
    en_words.append(en_word), en_vectors.append(en_wv.get_vector(en_word, norm=True))

ru_vectors, en_vectors = np.stack(ru_vectors), np.stack(en_vectors)
vectors = np.vstack([ru_vectors @ W.T, en_vectors])

pca1, pca2, pca3 = PCA(n_components=3).fit_transform(vectors).T

fig = px.scatter_3d(
    pd.DataFrame(
        {
            "PCA1": pca1,
            "PCA2": pca2,
            "PCA3": pca3,
            "Word": ru_words + en_words,
            "Lang": ["ru"] * N + ["en"] * N,
        }
    ),
    x="PCA1",
    y="PCA2",
    z="PCA3",
    color="Lang",
    hover_name="Word",
)
fig.update_layout(template="plotly_white", width=1000, height=800)
fig.update_traces(marker_size=5)
fig.show()

## **1.4. Анализ тональности текста. Кинопоиск**

Теперь давайте вернемся к задаче с прошлого семинара — классификации отзывов на позитивные и негативные. Мы изучили новый способ представлять тексты в виде эмбеддингов. Проверим, поможет ли он повысить качество модели.

In [ ]:
def load_kinopoisk_reviews(split: Literal["train", "validation", "test"]) -> pd.DataFrame:
    return pd.read_json(
        f"https://huggingface.co/datasets/ai-forever/kinopoisk-sentiment-classification/resolve/main/{split}.jsonl",
        lines=True,
    )


train = load_kinopoisk_reviews("train")
train.head()

In [ ]:
tokenized_train = [tokenize(text) for text in train["text"]]

wv = Word2Vec(
    sentences=tokenized_train,
    vector_size=128,
    window=5,
    min_count=5,
    max_vocab_size=100_000,
    seed=42,
    sg=0,
    negative=25,
    epochs=10,
).wv

tfidf = TfidfVectorizer(
    lowercase=False,
    preprocessor=None,
    tokenizer=lambda tokens: tokens,
    token_pattern=None,
    vocabulary=wv.key_to_index,
)

tfidf_train = tfidf.fit_transform(tokenized_train)

X_train = tfidf_train @ wv.vectors

pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
pipeline.fit(X_train, train["label_text"])

In [ ]:
test = load_kinopoisk_reviews("test")

tokenized_test = [tokenize(text) for text in test["text"]]

tfidf_test = tfidf.transform(tokenized_test)

X_test = tfidf_test @ wv.vectors
preds = pipeline.predict(X_test)

print(classification_report(test["label_text"], preds, digits=3, zero_division=0))

Получилось значительно хуже, чем можно было ожидать — даже хуже, чем у BoW, построенного нами ранее. И на что мы тогда вообще тратим свое время?

Давайте разбираться. Для этого посмотрим на косинусную близость нескольких слов.

In [ ]:
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

words = ["фильм", "кинематограф", "кино", "отлично", "хорошо", "ужасно", "плохо"]
vectors = np.vstack([wv[word] for word in words])

similarity = pd.DataFrame(
    cosine_similarity(vectors),
    index=words,
    columns=words,
)

sns.heatmap(
    similarity,
    vmin=-1,
    vmax=1,
    center=0,
    annot=True,
    fmt=".2f",
    square=True,
)

<div style="border-left: 4px solid dodgerblue; padding: 12px 12px 12px 24px; margin: 12px 0; color: dodgerblue; display: flex; align-items: center;">
    <strong>Крошка сын к отцу пришел, и спросила кроха:  — Что такое хорошо и что такое плохо?</strong>
</div>

Кажется, мы нашли проблему. В пространстве Word2Vec слова **хорошо** и **плохо** оказались гораздо ближе друг к другу, чем нам хотелось бы.

На самом деле в этом нет ничего удивительного. Слова с противоположным значением часто встречаются в очень похожих контекстах. Сравните:

- **Ужасный** фильм! Ничего **хуже** в жизни не видел!
- **Прекрасный** фильм! Ничего **лучше** в жизни не видел!

С точки зрения Word2Vec эти контексты практически одинаковы. Как следствие, антонимы вполне могут получить похожие векторные представления.

Таким образом, дистрибутивная гипотеза может сыграть с нами злую шутку при решении задачи анализа тональности текстов.

## **1.5. Заслуженный отдых**

Мы проделали долгий путь и заслужили отдых. Теперь давайте просто поиграем. Возьмем предобученные векторные представления для русского языка из библиотеки `navec` [[GitHub](https://github.com/natasha/navec)] (часть проекта [Natasha](https://natasha.github.io/)).

In [ ]:
# !wget https://storage.yandexcloud.net/natasha-navec/packs/navec_hudlit_v1_12B_500K_300d_100q.tar -P data

In [ ]:
from navec import Navec

navec = Navec.load("data/navec_hudlit_v1_12B_500K_300d_100q.tar")

In [ ]:
navec["навек"], navec["<pad>"]

In [ ]:
navec["<unk>"]

In [ ]:
from gensim.models import KeyedVectors

words = navec.vocab.words[: navec.vocab.unk_id]
vectors = navec.pq.unpack()[: navec.vocab.unk_id]

wv = KeyedVectors(vector_size=vectors.shape[1])
wv.add_vectors(words, vectors)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

tasks: list[tuple[str, str, str]] = [
    ("python", "боль", "счастье"),
    ("биолог", "природа", "компьютер"),
    ("макароны", "италия", "россия"),
    ("студент", "сон", "кофе"),
    ("учебник", "знания", "вымысел"),
    ("кентавр", "конь", "рыбка"),
    ("русалка", "рыба", "птица"),
    ("пегас", "крылья", "рог"),
    ("вампир", "мышь", "волк"),
    ("пушкин", "россия", "англия"),
    ("пушкин", "россия", "германия"),
    ("пушкин", "россия", "италия"),
]


def make_card(a: str, b: str, c: str) -> widgets.VBox:
    result = widgets.HTML("<b>???</b>")
    button = widgets.Button(description="Показать")

    def reveal(_: widgets.Button) -> None:
        if button.description == "Показать":
            word, _ = wv.most_similar(
                positive=[a, c],
                negative=[b],
                topn=1,
            )[0]
            result.value = f"<b>{word}</b>"
            button.description = "Скрыть"
        else:
            result.value = "<b>???</b>"
            button.description = "Показать"

    button.on_click(reveal)

    return widgets.VBox(
        [
            widgets.HTML(f"<b>{a} − {b} + {c}</b>"),
            result,
            button,
        ],
        layout=widgets.Layout(
            border="1px solid #ddd",
            padding="12px",
            width="280px",
            align_items="center",
        ),
    )


cards = [make_card(*task) for task in tasks]

display(
    widgets.GridBox(
        cards,
        layout=widgets.Layout(
            grid_template_columns="repeat(3, 280px)",
            grid_gap="12px",
        ),
    )
)